# VaR Comparison — Phase 2: Historical vs. Parametric

**Purpose:** Compare two methods of calculating VaR.

**METHOD 1: HISTORICAL**
- Sort past returns
- Take the percentile
- Assumption: past is prologue

**METHOD 2: PARAMETRIC (Normal Distribution)**
- Assume returns are normally distributed
- Use mean + Z-score + volatility
- Assumption: Returns follow a bell curve

**When methods agree:** confidence  
**When methods diverge:** danger signal (distribution assumptions breaking)

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import yfinance as yf
from datetime import datetime, timedelta

print("✅ Imports loaded")

## Utility Functions

In [ ]:
def download_price_data(ticker, days=252):
    """Download historical price data."""
    print(f"  Downloading {ticker}...")
    try:
        data = yf.download(ticker, period=f"{days}d", progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            return data.xs('Close', level=0, axis=1)[ticker]
        else:
            return data['Close']
    except Exception as e:
        print(f"  Error: {e}")
        raise

def calculate_returns(prices):
    """Calculate daily returns."""
    return prices.pct_change().dropna()

def get_portfolio_returns(tickers, weights, days=252):
    """Get weighted portfolio returns."""
    all_returns = {}
    for ticker in tickers:
        prices = download_price_data(ticker, days=days)
        returns = calculate_returns(prices)
        all_returns[ticker] = returns
    
    returns_df = pd.DataFrame(all_returns).dropna()
    portfolio_returns = (returns_df * weights).sum(axis=1)
    
    return portfolio_returns, returns_df

print("✅ Utility functions defined")

## Method 1: Historical VaR

In [ ]:
def var_historical(returns, confidence=0.95):
    """
    Historical VaR: Just take the percentile.
    
    Args:
        returns: Array of historical returns
        confidence: Confidence level (0.95 = worst 5%)
    
    Returns:
        VaR as a percentage
    """
    percentile = (1 - confidence) * 100  # For 95%, this is 5th percentile
    var_pct = np.percentile(returns, percentile)
    return var_pct

print("✅ Historical VaR defined")

## Method 2: Parametric VaR (Normal Distribution)

In [ ]:
def var_parametric(returns, confidence=0.95):
    """
    Parametric VaR: Assume normal distribution.
    
    Formula:
        VaR = Mean - (Z-score × Std Dev)
    
    Example:
        - Mean return: 0.05%
        - Volatility: 1.0%
        - 95% confidence Z-score: -1.6449 (negative because we want the downside)
        - VaR = 0.05% - (-1.6449 × 1.0%) = 0.05% + 1.6449% = -1.5949%
    
    Args:
        returns: Array of historical returns
        confidence: Confidence level
    
    Returns:
        VaR as a percentage
    """
    
    mean = np.mean(returns)
    std_dev = np.std(returns)
    
    # Get the Z-score for this confidence level
    # ppf = percent point function (inverse of CDF)
    # We want the lower tail, so (1 - confidence)
    z_score = stats.norm.ppf(1 - confidence)
    
    # Calculate VaR
    var_pct = mean + (z_score * std_dev)
    
    return var_pct, mean, std_dev, z_score

print("✅ Parametric VaR defined")

## Comparison Function

In [ ]:
def compare_var_methods(returns, confidence=0.95):
    """
    Compare historical vs parametric VaR.
    
    Returns: dict with both methods and diagnostics.
    """
    
    # Method 1: Historical
    var_hist = var_historical(returns, confidence=confidence)
    
    # Method 2: Parametric
    var_param, mean, std_dev, z_score = var_parametric(returns, confidence=confidence)
    
    # Calculate diagnostics
    abs_diff = var_param - var_hist
    rel_diff = abs_diff / abs(var_hist) * 100 if var_hist != 0 else 0
    
    # Check for normality (Shapiro-Wilk test)
    try:
        shapiro_stat, shapiro_p = stats.shapiro(returns[:5000] if len(returns) > 5000 else returns)
    except:
        shapiro_stat, shapiro_p = np.nan, np.nan
    
    # Skewness and kurtosis (measures of non-normality)
    skewness = stats.skew(returns)
    kurtosis = stats.kurtosis(returns)
    
    return {
        'var_historical': var_hist,
        'var_parametric': var_param,
        'difference': abs_diff,
        'relative_diff_pct': rel_diff,
        'mean': mean,
        'std_dev': std_dev,
        'z_score': z_score,
        'shapiro_stat': shapiro_stat,
        'shapiro_p': shapiro_p,
        'skewness': skewness,
        'kurtosis': kurtosis,
    }

print("✅ Comparison function defined")

## Reporting Function

In [ ]:
def print_comparison(comparison, portfolio_value=10_000_000, confidence=0.95):
    """Print a detailed comparison."""
    
    var_hist = comparison['var_historical']
    var_param = comparison['var_parametric']
    
    var_hist_dollar = var_hist * portfolio_value
    var_param_dollar = var_param * portfolio_value
    
    print(f"\n{'='*80}")
    print(f"VaR COMPARISON: Historical vs. Parametric ({confidence*100}% confidence)")
    print(f"{'='*80}\n")
    
    print(f"RESULTS:")
    print(f"  Historical VaR (%)      {var_hist*100:>10.4f}%    ${var_hist_dollar:>15,.0f}")
    print(f"  Parametric VaR (%)      {var_param*100:>10.4f}%    ${var_param_dollar:>15,.0f}")
    print(f"  Difference (%)          {comparison['difference']*100:>10.4f}%    ${comparison['difference']*portfolio_value:>15,.0f}")
    print(f"  Relative diff           {comparison['relative_diff_pct']:>10.2f}%")
    
    print(f"\nDISTRIBUTION DIAGNOSTICS:")
    print(f"  Mean return             {comparison['mean']*100:>10.4f}%")
    print(f"  Std deviation           {comparison['std_dev']*100:>10.4f}%")
    print(f"  Z-score ({confidence*100}%)       {comparison['z_score']:>10.4f}")
    print(f"  Skewness                {comparison['skewness']:>10.4f}  (0=normal, <0=left tail, >0=right tail)")
    print(f"  Excess kurtosis         {comparison['kurtosis']:>10.4f}  (0=normal, >0=fat tails)")
    
    print(f"\nNORMALITY TEST (Shapiro-Wilk):")
    if not np.isnan(comparison['shapiro_p']):
        if comparison['shapiro_p'] > 0.05:
            print(f"  p-value = {comparison['shapiro_p']:.4f} > 0.05 → Data looks NORMAL ✓")
        else:
            print(f"  p-value = {comparison['shapiro_p']:.4f} < 0.05 → Data is NOT normal ⚠️")
    else:
        print(f"  (Test skipped)")
    
    print(f"\nINTERPRETATION:")
    
    if abs(comparison['relative_diff_pct']) < 5:
        print(f"  ✓ Methods agree closely (<5% difference)")
        print(f"    → Parametric approximation is reasonable")
    elif abs(comparison['relative_diff_pct']) < 15:
        print(f"  ⚠️ Methods diverge moderately (5-15% difference)")
        print(f"    → Distribution assumptions matter")
    else:
        print(f"  🚨 Methods diverge significantly (>15% difference)")
        print(f"    → Distribution assumptions are VERY wrong")
    
    if comparison['kurtosis'] > 1:
        print(f"\n  ⚠️ Fat tails detected (excess kurtosis = {comparison['kurtosis']:.2f})")
        print(f"    → Consider Expected Shortfall (CVaR) instead")
    
    print(f"\n{'='*80}\n")

print("✅ Reporting function defined")

## Download Data & Compare

In [ ]:
print(f"\n{'='*80}")
print(f"STEP 1: Download portfolio data")
print(f"{'='*80}\n")

# 60/40 portfolio: 60% stocks (SPY), 40% bonds (BND)
tickers = ['SPY', 'BND']
weights = [0.6, 0.4]

portfolio_returns, individual_returns = get_portfolio_returns(
    tickers=tickers,
    weights=weights,
    days=252
)

print(f"\nLoaded {len(portfolio_returns)} trading days of data\n")

## Compare at Different Confidence Levels

In [ ]:
confidence_levels = [0.90, 0.95, 0.99]

for conf in confidence_levels:
    print(f"\n{'='*80}")
    print(f"CONFIDENCE LEVEL: {conf*100}%")
    print(f"{'='*80}")
    
    comparison = compare_var_methods(portfolio_returns, confidence=conf)
    print_comparison(comparison, portfolio_value=10_000_000, confidence=conf)

## Deep Dive: 95% Confidence

In [ ]:
print(f"\n{'='*80}")
print(f"DETAILED ANALYSIS (95% confidence)")
print(f"{'='*80}\n")

comparison_95 = compare_var_methods(portfolio_returns, confidence=0.95)

print(f"When parametric and historical diverge, it's a warning signal.")
print(f"It means the normal distribution assumption is breaking down.")
print(f"\nIn your 60/40 portfolio:")
print(f"  - Parametric assumes returns are perfectly normal")
print(f"  - Historical just looks at what actually happened")
print(f"  - The difference tells you how 'off' the assumption is")
print(f"\nFor risk management:")
print(f"  - If they agree → parametric is OK for quick estimates")
print(f"  - If they diverge → trust historical, investigate why")
print(f"  - If divergence is large → consider stress testing")